# 工具调用

期末快到了，小明问学习助手：“今天是几号？离 12 月 20 日的考试还有几天？”

学习助手的回答要么含糊其辞，要么干脆说错了日期。这不怪它：大模型的知识来自训练时读过的大量文字，训练一结束，它的知识就停在了那一刻。它没有日历，也没有时钟，更没法上网查。

大模型只会做一件事：**生成文字**。它不能读文件、不能查日期，也不能发邮件。

但我们的程序可以。那能不能让大模型“说”出它需要什么，再由程序去“做”呢？这就是**工具调用**（tool calling，也叫 function calling）。

## 工具调用是怎么工作的

工具调用的过程，就像小明和学习助手之间多了一位跑腿的帮手：

```
程序  → 大模型：小明问“今天是几号？”。另外，你可以使用这些工具：get_current_time（查当前时间）……
大模型 → 程序：  我需要调用工具 get_current_time，参数是 {"timezone": "America/Toronto"}
程序：          （真正运行 get_current_time 函数，得到 "2026-11-15 14:06:00 EST"）
程序  → 大模型：工具的结果是 "2026-11-15 14:06:00 EST"
大模型 → 程序：  今天是 2026 年 11 月 15 日，星期日。
```

这里有一点特别重要：**大模型从来不会自己去执行代码**。它只是“说出”想调用哪个工具、传什么参数，真正去执行的是我们的程序。执行的结果再交回给大模型，由它组织成最终的回答。

所以，要实现工具调用，程序需要做三件事：

1. **告诉大模型有哪些工具**：每个工具的名字、用途和参数。
2. **执行大模型请求的工具**：按它给的名字和参数，调用对应的 Python 函数。
3. **把结果交回去**：让大模型根据结果，给出最终的回答。

代码会多一些新东西，我们一样一样来看。

In [ ]:
import json
import os
import uuid
from abc import ABC, abstractmethod
from datetime import datetime
from enum import Enum
from pathlib import Path
from typing import Callable, List
from zoneinfo import ZoneInfo

import anthropic
from anthropic.types import UserMessageParam, AssistantMessageParam, ToolParam, TextBlockParam, ToolUseBlockParam, ToolResultBlockParam
from openai import OpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam, ChatCompletionAssistantMessageParam
from openai.types.chat import ChatCompletionToolParam, ChatCompletionMessageToolCallParam, ChatCompletionToolMessageParam

In [ ]:
def note(label: str, text: str):
    """Print one step of the agent's work, indented, so it stands apart from questions and answers."""
    lines = str(text).splitlines() or [""]
    print(f"  * {label}: {lines[0]}")
    for line in lines[1:]:
        print(f"    {line}")

## 新的消息类型

工具调用让对话里多了两种新东西：大模型发出的“工具调用请求”，和程序交回去的“工具结果”。消息类也要跟着扩展。

首先，角色里多了一个 `TOOL`，表示“工具的结果”：

In [ ]:
class MsgType(Enum):
    SYSTEM = "System"
    USER = "User"
    ASSISTANT = "Assistant"
    TOOL = "Tool"

In [ ]:
class Message(ABC):

    def __init__(self, msg_type: MsgType):
        self.msg_type = msg_type

    @abstractmethod
    def summary(self) -> str:
        pass

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "content": self.summary()}

    def __repr__(self) -> str:
        return f"Message({self.msg_type.value}): {self.summary()}"

In [ ]:
class SystemMessage(Message):

    def __init__(self, description: str):
        super().__init__(MsgType.SYSTEM)
        self.description = description

    def summary(self) -> str:
        return self.description

In [ ]:
class UserMessage(Message):

    def __init__(self, request: str):
        super().__init__(MsgType.USER)
        self.request = request

    def summary(self) -> str:
        return self.request

`ToolCall` 表示大模型发出的一次工具调用请求，包含三样东西：

- `id`：这次调用的编号，由大模型生成。大模型有时会一次请求好几个工具，交回结果时要用编号对上号。
- `name`：要调用的工具名字。
- `arguments`：调用参数，是一个字典。

`__repr__` 让它打印出来像一次函数调用，比如 `get_current_time({"timezone": "Asia/Tokyo"})`，方便我们看清楚发生了什么。

In [ ]:
class ToolCall:

    def __init__(self, id: str, name: str, arguments: dict):
        self.id = id
        self.name = name
        self.arguments = arguments

    def to_dict(self) -> dict:
        return {"id": self.id, "name": self.name, "arguments": self.arguments}

    def __repr__(self) -> str:
        return f"{self.name}({json.dumps(self.arguments, ensure_ascii=False)})"

大模型的回答，现在可能不是一段文字，而是一个或几个工具调用请求。所以 `AssistantMessage` 多了一个字段 `tool_calls`，存放这些请求（没有请求时就是空列表）。`to_dict` 也把它一起保存下来。

In [ ]:
class AssistantMessage(Message):

    def __init__(self, content: str, tool_calls: List[ToolCall] = None):
        super().__init__(MsgType.ASSISTANT)
        self.content = content
        self.tool_calls = tool_calls or []

    def summary(self) -> str:
        if self.tool_calls:
            return f"{self.content} [call: {self.tool_calls}]"
        return self.content

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "content": self.content,
                "tool_calls": [call.to_dict() for call in self.tool_calls]}

`ToolResultMessage` 表示工具执行的结果，它通过 `tool_call_id` 和对应的工具调用请求对上号。

In [ ]:
class ToolResultMessage(Message):

    def __init__(self, tool_call_id: str, name: str, result: str):
        super().__init__(MsgType.TOOL)
        self.tool_call_id = tool_call_id
        self.name = name
        self.result = result

    def summary(self) -> str:
        return f"{self.name} -> {self.result}"

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "tool_call_id": self.tool_call_id,
                "name": self.name, "content": self.result}

`message_from_dict` 也增加了这两种新消息的还原方法，这样带有工具调用的记忆，同样可以保存和加载：

In [ ]:
def message_from_dict(data: dict) -> Message:
    if data["type"] == MsgType.ASSISTANT.value:
        return AssistantMessage(data["content"], [ToolCall(**c) for c in data.get("tool_calls", [])])
    if data["type"] == MsgType.TOOL.value:
        return ToolResultMessage(data["tool_call_id"], data["name"], data["content"])
    classes = {MsgType.SYSTEM.value: SystemMessage, MsgType.USER.value: UserMessage}
    return classes[data["type"]](data["content"])

In [ ]:
system_message = SystemMessage("You're a learning assistant for the book 《从大模型到智能体》 (From LLM to Agent). "
                               "You help beginners learn how to build AI agents.")

## 定义一个工具

一个工具由四部分组成：

- `name`：工具的名字，大模型请求调用时就用这个名字。
- `description`：工具的用途。**大模型主要靠这段描述来决定什么时候用哪个工具**，所以一定要写清楚。
- `parameters`：参数的说明，用 **JSON Schema** 的格式来写。
- `function`：真正执行的 Python 函数。

`run` 方法用大模型给的参数字典调用这个函数。`**arguments` 会把字典展开成关键字参数，比如 `{"timezone": "Asia/Tokyo"}` 会变成 `function(timezone="Asia/Tokyo")`。结果统一转换成字符串，因为交回给大模型的只能是文字。

In [ ]:
class Tool:

    def __init__(self, name: str, description: str, parameters: dict, function: Callable):
        self.name = name
        self.description = description
        self.parameters = parameters  # JSON Schema of the arguments
        self.function = function

    def run(self, arguments: dict) -> str:
        return str(self.function(**arguments))

现在定义学习助手的第一个工具：查询当前的日期和时间。

函数 `get_current_time` 本身很简单：用 `datetime.now` 取出指定时区的当前时间。时区的名字用的是 IANA 格式，比如蒙特利尔所在的东部时区是 `America/Toronto`，北京时间是 `Asia/Shanghai`。

重点是 `parameters`，也就是 JSON Schema。它是一种用 JSON 描述数据格式的标准，在这里的意思是：

- `"type": "object"`：参数是一个对象（字典）；
- `"properties"`：它有一个字段 `timezone`，类型是字符串，`description` 解释了它是什么、该怎么填；
- `"required"`：`timezone` 是必须提供的。

大模型会读这份说明，按照它的要求生成参数。最后一行直接运行一下这个工具，确认它能正常工作。

In [ ]:
def get_current_time(timezone: str) -> str:
    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M:%S %Z")


time_tool = Tool(
    name="get_current_time",
    description="Get the current local date and time in a given IANA timezone.",
    parameters={
        "type": "object",
        "properties": {
            "timezone": {"type": "string", "description": "IANA timezone name, e.g. 'America/Toronto', 'Asia/Shanghai'"}
        },
        "required": ["timezone"]
    },
    function=get_current_time
)

print(time_tool.run({"timezone": "America/Toronto"}))

## 记忆的两处小改动

`Memory` 有两处改动，都是为了照顾工具调用：

- `add` 现在可以一次加入一整轮的多条消息（`*messages` 表示可以传入任意多个参数）。有了工具调用，一轮对话可能包含四五条消息：问题、工具调用请求、工具结果、最终回答。整轮加完再检查是否需要压缩，就不会在一轮对话的中间压缩。
- `summarize` 在切分时，保证**工具调用请求和它的结果不会被拆开**：从切分点往前找，直到遇到一条用户消息为止。如果只留下了结果、却把请求压缩进了摘要，API 会因为找不到对应的请求而报错。

In [ ]:
class Memory:
    SUMMARIZE_SYSTEM_MSG = "You are a conversation summary assistant."
    SUMMARIZE_USER_MSG = "Please condense the following conversation into a brief summary: "
    SUMMARY_PREFIX = "Historical Summary: "

    def __init__(self, provider, max_tokens=1000, max_messages=3, keep_messages=1, memory_id=None):
        self.provider = provider
        self.id = memory_id or uuid.uuid4().hex[:8]
        self.messages: List[Message] = []
        self.max_tokens = max_tokens
        self.max_messages = max_messages
        self.keep_messages = keep_messages

    def add(self, *messages: Message):
        # add a whole turn at once, so it is summarized (at most) once per turn
        self.messages.extend(messages)
        if len(self.messages) > self.max_messages or self.token_count() > self.max_tokens:
            self.summarize()

    def reset(self):
        self.messages = []

    def history(self, messages: List[Message] = None) -> str:
        return "\n".join(
            f"[{msg.msg_type.value}]: {msg.summary()}" for msg in (messages or self.messages)
        )

    def summarize(self):
        if self.keep_messages >= len(self.messages):
            return

        split = len(self.messages) - self.keep_messages
        # never separate a tool call from its result: keep from the last UserMessage onwards
        while split > 0 and not isinstance(self.messages[split], UserMessage):
            split -= 1
        if split == 0:
            return
        to_summarize = self.messages[:split]
        to_keep = self.messages[split:]

        request = UserMessage(Memory.SUMMARIZE_USER_MSG + "\n" + self.history(to_summarize))
        summary = self.provider.call(SystemMessage(Memory.SUMMARIZE_SYSTEM_MSG), [request])
        self.messages = [UserMessage(Memory.SUMMARY_PREFIX + summary.summary())] + to_keep
        note("summary", self.history())

    def token_count(self):
        # rough estimate: counts characters, not real tokens
        return sum(len(msg.summary() or "") for msg in self.messages)

    def to_dict(self) -> dict:
        return {"id": self.id, "messages": [msg.to_dict() for msg in self.messages]}

    @classmethod
    def from_dict(cls, data: dict, provider) -> "Memory":
        memory = cls(provider, memory_id=data["id"])
        memory.messages = [message_from_dict(m) for m in data["messages"]]
        return memory

In [ ]:
class MemoryStore:

    def __init__(self, root="memories"):
        self.root = Path(root)
        self.root.mkdir(exist_ok=True)

    def save(self, memory: Memory):
        path = self.root / f"{memory.id}.json"
        path.write_text(json.dumps(memory.to_dict(), ensure_ascii=False, indent=2), encoding="utf-8")

    def load(self, memory_id: str, provider) -> Memory:
        path = self.root / f"{memory_id}.json"
        if not path.exists():
            return None
        return Memory.from_dict(json.loads(path.read_text(encoding="utf-8")), provider)

## Provider：把工具告诉大模型

`call` 多了一个参数 `tools`。两家 API 都支持工具调用，但格式又不一样了：

| | OpenAI | Claude |
|---|---|---|
| 工具的格式 | `{"type": "function", "function": {name, description, parameters}}` | `{name, description, input_schema}` |
| 工具调用请求 | 助手消息里的 `tool_calls`，参数是一段 **JSON 字符串** | 助手消息内容里的 `tool_use` 块，参数是字典 |
| 工具结果 | 每个结果是一条单独的消息，角色是 `tool` | 结果是 `tool_result` 块，**同一轮的所有结果放进同一条用户消息** |

这些差别全都藏在了 Provider 里：

- `_tool_schema`：把我们的 `Tool` 翻译成各家要求的格式。
- `_message_schema`：增加了工具调用请求和工具结果的翻译。
- `_response_schema`：从回答里取出工具调用请求，变成我们的 `ToolCall`。OpenAI 的参数是 JSON 字符串，要先用 `json.loads` 转成字典。

In [ ]:
class LLMProvider(ABC):

    @abstractmethod
    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=4096):
        pass

In [ ]:
class OpenAIProvider(LLMProvider):

    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.default_model = "gpt-5.4-mini"

    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=4096):
        if system:
            messages = [system] + messages
        kwargs = dict(
            model=model or self.default_model,
            max_completion_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if tools:
            kwargs["tools"] = self._tool_schema(tools)
        response = self.client.chat.completions.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _tool_schema(tools: List[Tool]):
        return [
            ChatCompletionToolParam(type="function", function={
                "name": tool.name, "description": tool.description, "parameters": tool.parameters})
            for tool in tools
        ]

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                schema.append(ChatCompletionSystemMessageParam(role="system", content=msg.description))
            elif isinstance(msg, UserMessage):
                schema.append(ChatCompletionUserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                param = ChatCompletionAssistantMessageParam(role="assistant", content=msg.content)
                if msg.tool_calls:
                    param["tool_calls"] = [
                        ChatCompletionMessageToolCallParam(id=call.id, type="function", function={
                            "name": call.name, "arguments": json.dumps(call.arguments)})
                        for call in msg.tool_calls
                    ]
                schema.append(param)
            elif isinstance(msg, ToolResultMessage):
                # OpenAI: one "tool" message per tool result
                schema.append(ChatCompletionToolMessageParam(role="tool", tool_call_id=msg.tool_call_id, content=msg.result))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        message = response.choices[0].message
        tool_calls = [
            ToolCall(call.id, call.function.name, json.loads(call.function.arguments))
            for call in (message.tool_calls or [])
        ]
        return AssistantMessage(message.content or "", tool_calls)

In [ ]:
class ClaudeProvider(LLMProvider):

    def __init__(self):
        self.client = anthropic.Anthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
        self.default_model = "claude-haiku-4-5"

    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=4096):
        kwargs = dict(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if system:
            kwargs["system"] = system.summary()
        if tools:
            kwargs["tools"] = self._tool_schema(tools)
        response = self.client.messages.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _tool_schema(tools: List[Tool]):
        return [
            ToolParam(name=tool.name, description=tool.description, input_schema=tool.parameters)
            for tool in tools
        ]

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, UserMessage):
                schema.append(UserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                content = [TextBlockParam(type="text", text=msg.content)] if msg.content else []
                content += [
                    ToolUseBlockParam(type="tool_use", id=call.id, name=call.name, input=call.arguments)
                    for call in msg.tool_calls
                ]
                schema.append(AssistantMessageParam(role="assistant", content=content))
            elif isinstance(msg, ToolResultMessage):
                # Claude: all tool results of one turn go into a single "user" message
                block = ToolResultBlockParam(type="tool_result", tool_use_id=msg.tool_call_id, content=msg.result)
                if schema and schema[-1]["role"] == "user" and isinstance(schema[-1]["content"], list):
                    schema[-1]["content"].append(block)
                else:
                    schema.append(UserMessageParam(role="user", content=[block]))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        text = "".join(block.text for block in response.content if block.type == "text")
        tool_calls = [
            ToolCall(block.id, block.name, block.input)
            for block in response.content if block.type == "tool_use"
        ]
        return AssistantMessage(text, tool_calls)

## Agent：执行工具，再问一次

`Agent` 的变化有两处：

1. 创建时可以传入一组工具 `tools`，按名字存进字典，方便以后按名字找到工具。
2. 每一轮对话，按这个顺序进行：
   - 把问题和工具列表一起交给大模型；
   - 如果大模型的回答里有工具调用请求，就**逐个执行**，把每个结果包装成 `ToolResultMessage`；
   - 带着工具结果，**再调用一次**大模型，得到最终的回答；
   - 把这一整轮的消息（`turn`）一起存进记忆。

这一章只处理**一次**工具调用：如果大模型拿到结果以后，还想再调用工具，我们只能先提示一下。下一章会解决这个问题。

In [ ]:
class Agent:

    def __init__(self, provider: LLMProvider, system: SystemMessage, tools: List[Tool] = None):
        self.provider = provider
        self.system = system
        self.tools = {tool.name: tool for tool in (tools or [])}
        self.store = MemoryStore()
        self.memory = Memory(provider)

    def chat(self, questions: List[str] = None, system: SystemMessage = None):
        questions = iter(questions) if questions is not None else None  # preset questions, or None to type
        note("memory", f"#{self.memory.id}")
        while True:
            message = next(questions, "/quit") if questions else input("YOU: ")
            print(f"\nYOU: {message}")
            if message.strip().lower() == "/quit":
                break
            if not message.strip():
                continue

            if message.strip().lower() == "/new":
                self.memory = Memory(self.provider)
                note("new memory", f"#{self.memory.id}")
                continue

            if message.startswith("#"):
                memory = self.store.load(message[1:].strip(), self.provider)
                if memory is None:
                    note("memory", f"{message} not found")
                else:
                    self.memory = memory
                    note(f"loaded memory #{memory.id}", memory.history())
                continue

            turn = [UserMessage(message)]
            tools = list(self.tools.values())
            response = self.provider.call(system or self.system, self.memory.messages + turn, tools)
            turn.append(response)

            if response.tool_calls:
                for call in response.tool_calls:
                    result = self.tools[call.name].run(call.arguments)
                    note("tool", f"{call} -> {result}")
                    turn.append(ToolResultMessage(call.id, call.name, result))
                response = self.provider.call(system or self.system, self.memory.messages + turn, tools)
                if response.tool_calls:
                    print("(the model wants more tool calls - chapter 10 will handle it)")
                    response.tool_calls = []
                turn.append(response)

            self.memory.add(*turn)
            self.store.save(self.memory)
            print(f"AI: {response.content}")

## 问问今天几号

给学习助手装上查时间的工具，再问一次：

In [ ]:
provider = OpenAIProvider()
agent = Agent(provider, system_message, tools=[time_tool])
agent.chat(["今天是几号、星期几？"])
# or call agent.chat() without questions to type your own

缩进的 `* tool` 那一行，就是程序替大模型执行的工具调用。看看这一轮在记忆里是什么样子：

In [ ]:
print(agent.memory.history())

一轮对话变成了四条消息：小明的问题、大模型的工具调用请求、工具的结果，以及大模型的最终回答。

## 动手试试

- 换几种问法，比如“东京现在几点？”“我在北京，现在是上午还是下午？”，观察大模型给工具传了什么参数。
- 问一个不需要工具的问题，比如“什么是变量？”，看看大模型会不会调用工具。
- 再写一个工具，比如 `count_characters(text)`：统计一段文字有多少个字。记得写好 `description` 和 `parameters`，然后把它和 `time_tool` 一起交给学习助手。
- 问“离 12 月 20 日还有几天？”。如果大模型拿到日期以后还想再调用工具，就会看到那句提示。

## 小结

- 大模型只会生成文字，**工具调用**让它可以“说”出需要调用的工具和参数，由程序去执行，再把结果交回给它。大模型从不自己执行代码。
- 一个工具包括名字、描述、参数说明（JSON Schema）和真正执行的函数。描述决定了大模型什么时候用它。
- 消息里多了 `ToolCall`（工具调用请求）和 `ToolResultMessage`（工具结果），用 `id` 对上号。
- OpenAI 和 Claude 的工具调用格式差别不小，全都由 Provider 负责翻译。
- 记忆一次存入一整轮，压缩时不会把工具调用请求和结果拆开。

现在的学习助手只能调用一轮工具。可有些问题需要好几步才能完成，比如先查今天的日期，再算 30 天后是哪天。下一章，我们把“只调用一次”变成一个**循环**。